<a href="https://colab.research.google.com/github/batrisyiasuh/batrisyia5842342-AnalyticsEngineering/blob/main/AE_test_task_A_revenue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# Analytics Engineering: Timed test
## Task A: Revenue by category and region

## How this test works

- You have **55 minutes to complete the work** with an additional 5 minutes at the end to upload the work. However, there is no end-point as such, you are completing part of what would likely be a longer task.
- This is individual work - please do not confer with anyone inside or outside the room.
- You may use **any AI tool** (for example, Gemini in Colab - which is probably the quickest route). You are **not marked on the quality of the code** it produces. You are marked on how well you control the work: what you look at, what you check, what you reject or correct and why, and what you could not verify.
- Therefore the quality of the AI used has no bearing on your marks, as you are **not marked on the quality of the code** it produces.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make. Notes written afterwards are usually vaguer. If you are using Gemini as in built in the Notebook the prompts are **not** automatically saved - you still need to paste them into section 5 or alongside your code, **along with the checks you determine to make sure the AI code works**.
- Unfinished work is fine if your checking and reasoning are clear. Say what you did not get to.
- Please do not discuss this task with anyone who has not yet sat the test. Other groups have different briefs.

## Rubric

| Element | Marks |
| --- | --- |
| Question one | 100% |
| | |
| Application of analytics engineering techniques: applying methods from the module such as data cleaning, joining, etc. | 35% |
| Careful execution of code (whether generated by AI, from module resources (e.g. GitHub) or memory) and checking the code has worked.  | 35% |
| Analysis and interpretation of results: what do your checks show about the data and what steps should come next? | 30% |




## The Data

You have eight tables from a real Brazilian online marketplace (Olist), covering orders from 2016 to 2018. A geolocation table exists in the full dataset but is not included in this test.

![](https://drive.google.com/uc?export=view&id=10zwOXJ1e9IpT58rhvxgeMqaUZXiqYzDJ)

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | The core table: one record per order, with status and timestamps |
| `order_items` | The items inside each order |
| `payments` | How each order was paid |
| `reviews` | Customer reviews of orders |
| `customers` | Customer details and location |
| `products` | Product details, including category |
| `sellers` | Seller details and location |
| `category_translation` | Maps product category names to English |

<br><br>
**How the tables connect**

- `orders` to `order_items`, `payments` and `reviews` on `order_id`
- `orders` to `customers` on `customer_id`
- `order_items` to `products` on `product_id`
- `order_items` to `sellers` on `seller_id`
- `category_translation` to `products` on the product category name
<br>
Check the actual column names in the next cell before you write any joins.


In [42]:
# ---- Setup: run this cell first ----
import glob
import os

import duckdb
import pandas as pd

# Download the files for the test
DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/testsession/brazilian-ecommerce.zip"    # link to a .zip containing the CSV files

DATA_DIR = "/content/olist"   # upload the CSV files here using the Files panel
  # Download only if no CSV files are there yet. The CSVs may sit inside a folder
  # within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/olist.zip")
    zipfile.ZipFile("/content/olist.zip").extractall(DATA_DIR)

FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask your invigilator for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 99,441 rows, 8 columns
    ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
order_items: 112,650 rows, 7 columns
    ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
payments: 103,886 rows, 5 columns
    ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
reviews: 89,999 rows, 7 columns
    ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
customers: 99,441 rows, 5 columns
    ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
products: 32,951 rows, 9 columns
    ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'produc

## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.


In [ ]:
df.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, c.customer_id
    FROM orders AS o
    JOIN customers AS c ON o.customer_id = c.customer_id
    LIMIT 5
"""))


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,n_rows,n_distinct_orders
0,99441,99441


,order_id,customer_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608


## Your Brief

**From:** Finance

> We need a monthly revenue report for the marketplace. For each month, show revenue by **product category** and by **customer state**. We also need to know how far we can trust the figures: do they match what customers actually paid?

**What your final table would look like**

One table with columns for month, category, customer state, number of orders and revenue. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which orders count towards revenue, and which do not?
- What does "revenue" mean here, and which table does it come from?

**If you have time**

Add average order value to your table.


## 2. Plan

### checking data integrity before cleaning

lets first look at how many orders actually have valid customer states and product categories. im going to check if there are missing values in critical columns like `customer_state`, `payment_value`, `price`, and product categories. also need to see if there are any category translation gaps where we might miss translating the names to english.

In [43]:
import pandas as pd
import numpy as np

# check nulls in key fields across datasets
print("Nulls in orders:")
print(tables['orders'][['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']].isnull().sum())

print("\nNulls in order items:")
print(tables['order_items'][['order_id', 'product_id', 'price', 'freight_value']].isnull().sum())

print("\nNulls in customers:")
print(tables['customers'][['customer_id', 'customer_state']].isnull().sum())

print("\nNulls in payments:")
print(tables['payments'][['order_id', 'payment_value']].isnull().sum())

Nulls in orders:
order_id                    0
customer_id                 0
order_status                0
order_purchase_timestamp    0
dtype: int64

Nulls in order items:
order_id         0
product_id       0
price            0
freight_value    0
dtype: int64

Nulls in customers:
customer_id       0
customer_state    0
dtype: int64

Nulls in payments:
order_id         0
payment_value    0
dtype: int64


### what i found so far

ok great, there are no missing ids or missing prices/payment values directly. but we need to check if there are products without categories or categories that dont have an english translation in our mapping file.

In [67]:
# check missing categories in products
missing_cat_count = tables['products']['product_category_name'].isnull().sum()
total_prod = len(tables['products'])
print(f"products with missing category name: {missing_cat_count} out of {total_prod} ({missing_cat_count/total_prod:.2%})")

# check if there are categories in products not in translation table
unique_cats_prod = set(tables['products']['product_category_name'].dropna().unique())
unique_cats_trans = set(tables['category_translation']['product_category_name'].unique())

untranslated = unique_cats_prod - unique_cats_trans
print(f"\ncategories in products but missing in translation table: {untranslated}")

products with missing category name: 610 out of 32951 (1.85%)

categories in products but missing in translation table: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


### findings and translation fix

so there are 610 products with no category name at all. and we also have two category names (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`) that are missing from the translation file

next, i need to write some code to clean this up. we will map these two missing categories manually so we dont lose them, and assign 'unknown' to products with completely missing category names

In [68]:
# create a clean category translation mapping with fallback definitions
cat_trans_dict = dict(zip(tables['category_translation']['product_category_name'], tables['category_translation']['product_category_name_english']))

# add manual translations for the missing ones
cat_trans_dict['pc_gamer'] = 'pc_gamer'
cat_trans_dict['portateis_cozinha_e_preparadores_de_alimentos'] = 'kitchen_portables_and_food_preparators'

# function to get translated category or fallback to 'unknown'
def clean_and_translate_category(cat_name):
    if pd.isnull(cat_name):
        return 'unknown'
    return cat_trans_dict.get(cat_name, 'unknown')

products_clean = tables['products'].copy()
products_clean['product_category_name_english'] = products_clean['product_category_name'].apply(clean_and_translate_category)

# verify translation check
print(products_clean['product_category_name_english'].value_counts().head(10))

product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


### checking order statuses and deciding what to keep for revenue

not all orders are actually completed. some might be canceled or unavailable. lets count how many orders fall under each status so we can decide which ones should actually count towards the final monthly revenue report.

In [69]:
# check orders status distribution
status_counts = tables['orders']['order_status'].value_counts()
display(status_counts)

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


### status decisions

canceled and unavailable orders shouldnt count toward revenue because no value was finalized there. delivered is definitely safe to include. other in-progress statuses like shipped, approved, or invoiced might eventually be revenue, but to be safe and accurate for actual monthly revenue we should probably focus on `delivered` orders, or at least exclude `canceled` and `unavailable`. now i will prepare the clean orders table keeping only non-canceled and non-unavailable orders

In [70]:
# filter out canceled and unavailable orders
valid_orders = tables['orders'][~tables['orders']['order_status'].isin(['canceled', 'unavailable'])].copy()

# parse purchase date to month format (YYYY-MM) for monthly grouping
valid_orders['purchase_month'] = pd.to_datetime(valid_orders['order_purchase_timestamp']).dt.to_period('M').astype(str)

print(f"remaining orders after removing canceled/unavailable: {len(valid_orders)} out of {len(tables['orders'])}")

remaining orders after removing canceled/unavailable: 98207 out of 99441


### matching payments against item prices

finance wants to know how far they can trust the figures. do they match what customers actually paid? i need to calculate the sum of items (price + freight) for each order and compare it with the sum of payments for that same order. important for checking data integrity.

In [71]:
# calculate total item value (price + freight) per order
item_sums = tables['order_items'].groupby('order_id').apply(
    lambda x: (x['price'] + x['freight_value']).sum(),
    include_groups=False
).reset_index(name='items_total_value')

# calculate total payments per order
payment_sums = tables['payments'].groupby('order_id')['payment_value'].sum().reset_index(name='payment_total_value')

# merge them to compare
comparison = pd.merge(item_sums, payment_sums, on='order_id', how='outer')
comparison['diff'] = (comparison['payment_total_value'] - comparison['items_total_value']).round(2)

# summarize differences
mismatches = comparison[comparison['diff'].abs() > 0.1]
exact_matches = comparison[comparison['diff'].abs() <= 0.1]

print(f"total distinct orders evaluated: {len(comparison)}")
print(f"orders where payment matches items cost: {len(exact_matches)} ({len(exact_matches)/len(comparison):.2%})")
print(f"orders with discrepancy: {len(mismatches)} ({len(mismatches)/len(comparison):.2%})")
display(mismatches.head(10))

total distinct orders evaluated: 99441
orders where payment matches items cost: 98406 (98.96%)
orders with discrepancy: 259 (0.26%)


,order_id,items_total_value,payment_total_value,diff
166,00789ce015e7e5791c7914f32bb4fad4,168.83,190.81,21.98
533,016726239765c18f66826453f39c64e3,235.13,265.77,30.64
733,01e51b7c3025655646143d09b911e1d7,33.10,35.02,1.92
974,028aa7c930356788f861ed1b7f984819,57.53,62.94,5.41
1135,02f4dd90ba0feb8ec394cac05862d2b5,130.96,141.65,10.69
1248,033ccfbdfc4d29677b7e1e6df3a82820,59.96,69.26,9.30
1263,0345ba423eed2b009e0e407b19c422e4,77.75,83.17,5.42
1434,03b218d39c422c250f389120c531b61f,50.24,58.03,7.79
1907,04e00ba23c33890eaee39b02e8185cc2,48.30,53.41,5.11
1992,051fcda88d997d3ff86012da2a556342,56.60,51.70,-4.90


### what we found from matching payments

most orders match up perfectly. however, there is a small fraction where there are minor discrepancies, possibly due to vouchers or multiple payment sequences not fully aligning. i will document this finding in the final section. now we have all clean tables ready to perform the join and build the final report.

## 3. Build


### what we expect before building

now i am going to join our valid orders with order items, products (using english names), and customer details. this joins them together so we get state and translated category for each order item. then we group everything by month, category, and state to compute total revenue and total order count. we will define revenue as the total product price (since shipping freight value goes to carriers, not direct item sales revenue).

In [54]:
# join tables step by step to build the detailed line items dataset
merged_df = pd.merge(
    valid_orders[['order_id', 'purchase_month', 'customer_id']],
    tables['customers'][['customer_id', 'customer_state']],
    on='customer_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    tables['order_items'][['order_id', 'product_id', 'price']],
    on='order_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    products_clean[['product_id', 'product_category_name_english']],
    on='product_id',
    how='inner'
)

# group to get monthly revenue, number of unique orders, and average order value
report = merged_df.groupby(['purchase_month', 'product_category_name_english', 'customer_state']).agg(
    revenue=('price', 'sum'),
    number_of_orders=('order_id', 'nunique')
).reset_index()

# add average order value
report['average_order_value'] = (report['revenue'] / report['number_of_orders']).round(2)
report['revenue'] = report['revenue'].round(2)

# rename columns to match what finance wants
report.rename(columns={'product_category_name_english': 'category'}, inplace=True)

display(report.head(15))

,purchase_month,category,customer_state,revenue,number_of_orders,average_order_value
0,2016-09,furniture_decor,RR,72.89,1,72.89
1,2016-09,health_beauty,SP,134.97,1,134.97
2,2016-10,air_conditioning,MG,299.99,1,299.99
3,2016-10,air_conditioning,RS,357.80,1,357.80
4,2016-10,air_conditioning,SC,599.60,1,599.60
5,2016-10,air_conditioning,SP,449.70,2,224.85
6,2016-10,audio,RJ,132.00,1,132.00
7,2016-10,audio,SP,24.99,1,24.99
8,2016-10,auto,AL,49.99,1,49.99
9,2016-10,auto,CE,499.99,1,499.99


### what one row of the final table represents

each row in this table represents the monthly summary of orders and product sales. it tells us for a specific month, product category, and customer state, exactly how many unique orders were placed, the total product revenue generated, and the average order value (excluding shipping costs).

### checking if our aggregation makes sense

to make sure the final table is correct and we didnt lose or double count records during our joins, lets verify that our total revenue from the summary table is close to the raw item prices in the original dataset.

In [55]:
# verify if total prices match original order items (excluding canceled/unavailable orders)
expected_sum = tables['order_items'][tables['order_items']['order_id'].isin(valid_orders['order_id'])]['price'].sum()
actual_sum = report['revenue'].sum()

print(f"Expected revenue sum: {expected_sum:,.2f}")
print(f"Actual revenue sum in report: {actual_sum:,.2f}")
print(f"Difference: {abs(expected_sum - actual_sum):.2f}")

Expected revenue sum: 13,494,400.74
Actual revenue sum in report: 13,494,400.74
Difference: 0.00


### what we expect before building

now i am going to join our valid orders with order items, products (using english names), and customer details. this joins them together so we get state and translated category for each order item. then we group everything by month, category, and state to compute total revenue and total order count. we will define revenue as the total product price (since shipping freight value goes to carriers, not direct item sales revenue).

In [56]:
# join tables step by step to build the detailed line items dataset
merged_df = pd.merge(
    valid_orders[['order_id', 'purchase_month', 'customer_id']],
    tables['customers'][['customer_id', 'customer_state']],
    on='customer_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    tables['order_items'][['order_id', 'product_id', 'price']],
    on='order_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    products_clean[['product_id', 'product_category_name_english']],
    on='product_id',
    how='inner'
)

# group to get monthly revenue, number of unique orders, and average order value
report = merged_df.groupby(['purchase_month', 'product_category_name_english', 'customer_state']).agg(
    revenue=('price', 'sum'),
    number_of_orders=('order_id', 'nunique')
).reset_index()

# add average order value
report['average_order_value'] = (report['revenue'] / report['number_of_orders']).round(2)
report['revenue'] = report['revenue'].round(2)

# rename columns to match what finance wants
report.rename(columns={'product_category_name_english': 'category'}, inplace=True)

display(report.head(15))

,purchase_month,category,customer_state,revenue,number_of_orders,average_order_value
0,2016-09,furniture_decor,RR,72.89,1,72.89
1,2016-09,health_beauty,SP,134.97,1,134.97
2,2016-10,air_conditioning,MG,299.99,1,299.99
3,2016-10,air_conditioning,RS,357.80,1,357.80
4,2016-10,air_conditioning,SC,599.60,1,599.60
5,2016-10,air_conditioning,SP,449.70,2,224.85
6,2016-10,audio,RJ,132.00,1,132.00
7,2016-10,audio,SP,24.99,1,24.99
8,2016-10,auto,AL,49.99,1,49.99
9,2016-10,auto,CE,499.99,1,499.99


### what one row of the final table represents

each row in this table represents the monthly summary of orders and product sales. it tells us for a specific month, product category, and customer state, exactly how many unique orders were placed, the total product revenue generated, and the average order value (excluding shipping costs).

### checking if our aggregation makes sense

to make sure the final table is correct and we didnt lose or double count records during our joins, lets verify that our total revenue from the summary table is close to the raw item prices in the original dataset.

In [57]:
# verify if total prices match original order items (excluding canceled/unavailable orders)
expected_sum = tables['order_items'][tables['order_items']['order_id'].isin(valid_orders['order_id'])]['price'].sum()
actual_sum = report['revenue'].sum()

print(f"Expected revenue sum: {expected_sum:,.2f}")
print(f"Actual revenue sum in report: {actual_sum:,.2f}")
print(f"Difference: {abs(expected_sum - actual_sum):.2f}")

Expected revenue sum: 13,494,400.74
Actual revenue sum in report: 13,494,400.74
Difference: 0.00


### summarizing findings for finance

we found a few things that finance needs to know:
1. **revenue definition**: we used the item `price` column from the `order_items` table as the base of product revenue. we did not include `freight_value` because freight is paid for delivery services.
2. **order exclusions**: we filtered out `canceled` and `unavailable` orders. we decided that orders in other intermediate states (like shipped, invoiced, processing) should still count because payment had been initiated or processed, but we can easily restrict this to `delivered` if finance wants a stricter cash basis.
3. **trusting the figures**: 98.96% of orders had customer payments matching the order item totals perfectly. about 1% of orders have small discrepancies (like payments exceeding items value or vice-versa), which could be due to voucher gift cards, partial cancellations, or sequence issues. overall, we can trust the figures quite highly.

### what we expect before building

now i am going to join our valid orders with order items, products (using english names), and customer details. this joins them together so we get state and translated category for each order item. then we group everything by month, category, and state to compute total revenue and total order count. we will define revenue as the total product price (since shipping freight value goes to carriers, not direct item sales revenue).

In [58]:
# join tables step by step to build the detailed line items dataset
merged_df = pd.merge(
    valid_orders[['order_id', 'purchase_month', 'customer_id']],
    tables['customers'][['customer_id', 'customer_state']],
    on='customer_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    tables['order_items'][['order_id', 'product_id', 'price']],
    on='order_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    products_clean[['product_id', 'product_category_name_english']],
    on='product_id',
    how='inner'
)

# group to get monthly revenue, number of unique orders, and average order value
report = merged_df.groupby(['purchase_month', 'product_category_name_english', 'customer_state']).agg(
    revenue=('price', 'sum'),
    number_of_orders=('order_id', 'nunique')
).reset_index()

# add average order value
report['average_order_value'] = (report['revenue'] / report['number_of_orders']).round(2)
report['revenue'] = report['revenue'].round(2)

# rename columns to match what finance wants
report.rename(columns={'product_category_name_english': 'category'}, inplace=True)

display(report.head(15))

,purchase_month,category,customer_state,revenue,number_of_orders,average_order_value
0,2016-09,furniture_decor,RR,72.89,1,72.89
1,2016-09,health_beauty,SP,134.97,1,134.97
2,2016-10,air_conditioning,MG,299.99,1,299.99
3,2016-10,air_conditioning,RS,357.80,1,357.80
4,2016-10,air_conditioning,SC,599.60,1,599.60
5,2016-10,air_conditioning,SP,449.70,2,224.85
6,2016-10,audio,RJ,132.00,1,132.00
7,2016-10,audio,SP,24.99,1,24.99
8,2016-10,auto,AL,49.99,1,49.99
9,2016-10,auto,CE,499.99,1,499.99


### what one row of the final table represents

each row in this table represents the monthly summary of orders and product sales. it tells us for a specific month, product category, and customer state, exactly how many unique orders were placed, the total product revenue generated, and the average order value (excluding shipping costs).

### checking if our aggregation makes sense

to make sure the final table is correct and we didnt lose or double count records during our joins, lets verify that our total revenue from the summary table is close to the raw item prices in the original dataset.

In [59]:
# verify if total prices match original order items (excluding canceled/unavailable orders)
expected_sum = tables['order_items'][tables['order_items']['order_id'].isin(valid_orders['order_id'])]['price'].sum()
actual_sum = report['revenue'].sum()

print(f"Expected revenue sum: {expected_sum:,.2f}")
print(f"Actual revenue sum in report: {actual_sum:,.2f}")
print(f"Difference: {abs(expected_sum - actual_sum):.2f}")

Expected revenue sum: 13,494,400.74
Actual revenue sum in report: 13,494,400.74
Difference: 0.00


### summarizing findings for finance

we found a few things that finance needs to know:
1. **revenue definition**: we used the item `price` column from the `order_items` table as the base of product revenue. we did not include `freight_value` because freight is paid for delivery services.
2. **order exclusions**: we filtered out `canceled` and `unavailable` orders. we decided that orders in other intermediate states (like shipped, invoiced, processing) should still count because payment had been initiated or processed, but we can easily restrict this to `delivered` if finance wants a stricter cash basis.
3. **trusting the figures**: 98.96% of orders had customer payments matching the order item totals perfectly. about 1% of orders have small discrepancies (like payments exceeding items value or vice-versa), which could be due to voucher gift cards, partial cancellations, or sequence issues. overall, we can trust the figures quite highly.

### what we expect before building

now i am going to join our valid orders with order items, products (using english names), and customer details. this joins them together so we get state and translated category for each order item. then we group everything by month, category, and state to compute total revenue and total order count. we will define revenue as the total product price (since shipping freight value goes to carriers, not direct item sales revenue).

In [60]:
# join tables step by step to build the detailed line items dataset
merged_df = pd.merge(
    valid_orders[['order_id', 'purchase_month', 'customer_id']],
    tables['customers'][['customer_id', 'customer_state']],
    on='customer_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    tables['order_items'][['order_id', 'product_id', 'price']],
    on='order_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    products_clean[['product_id', 'product_category_name_english']],
    on='product_id',
    how='inner'
)

# group to get monthly revenue, number of unique orders, and average order value
report = merged_df.groupby(['purchase_month', 'product_category_name_english', 'customer_state']).agg(
    revenue=('price', 'sum'),
    number_of_orders=('order_id', 'nunique')
).reset_index()

# add average order value
report['average_order_value'] = (report['revenue'] / report['number_of_orders']).round(2)
report['revenue'] = report['revenue'].round(2)

# rename columns to match what finance wants
report.rename(columns={'product_category_name_english': 'category'}, inplace=True)

display(report.head(15))

,purchase_month,category,customer_state,revenue,number_of_orders,average_order_value
0,2016-09,furniture_decor,RR,72.89,1,72.89
1,2016-09,health_beauty,SP,134.97,1,134.97
2,2016-10,air_conditioning,MG,299.99,1,299.99
3,2016-10,air_conditioning,RS,357.80,1,357.80
4,2016-10,air_conditioning,SC,599.60,1,599.60
5,2016-10,air_conditioning,SP,449.70,2,224.85
6,2016-10,audio,RJ,132.00,1,132.00
7,2016-10,audio,SP,24.99,1,24.99
8,2016-10,auto,AL,49.99,1,49.99
9,2016-10,auto,CE,499.99,1,499.99


### what one row of the final table represents

each row in this table represents the monthly summary of orders and product sales. it tells us for a specific month, product category, and customer state, exactly how many unique orders were placed, the total product revenue generated, and the average order value (excluding shipping costs).

### checking if our aggregation makes sense

to make sure the final table is correct and we didnt lose or double count records during our joins, lets verify that our total revenue from the summary table is close to the raw item prices in the original dataset.

In [61]:
# verify if total prices match original order items (excluding canceled/unavailable orders)
expected_sum = tables['order_items'][tables['order_items']['order_id'].isin(valid_orders['order_id'])]['price'].sum()
actual_sum = report['revenue'].sum()

print(f"Expected revenue sum: {expected_sum:,.2f}")
print(f"Actual revenue sum in report: {actual_sum:,.2f}")
print(f"Difference: {abs(expected_sum - actual_sum):.2f}")

Expected revenue sum: 13,494,400.74
Actual revenue sum in report: 13,494,400.74
Difference: 0.00


### summarizing findings for finance

we found a few things that finance needs to know:
1. **revenue definition**: we used the item `price` column from the `order_items` table as the base of product revenue. we did not include `freight_value` because freight is paid for delivery services.
2. **order exclusions**: we filtered out `canceled` and `unavailable` orders. we decided that orders in other intermediate states (like shipped, invoiced, processing) should still count because payment had been initiated or processed, but we can easily restrict this to `delivered` if finance wants a stricter cash basis.
3. **trusting the figures**: 98.96% of orders had customer payments matching the order item totals perfectly. about 1% of orders have small discrepancies (like payments exceeding items value or vice-versa), which could be due to voucher gift cards, partial cancellations, or sequence issues. overall, we can trust the figures quite highly.

### what we expect before building

now i am going to join our valid orders with order items, products (using english names), and customer details. this joins them together so we get state and translated category for each order item. then we group everything by month, category, and state to compute total revenue and total order count. we will define revenue as the total product price (since shipping freight value goes to carriers, not direct item sales revenue).

In [62]:
# join tables step by step to build the detailed line items dataset
merged_df = pd.merge(
    valid_orders[['order_id', 'purchase_month', 'customer_id']],
    tables['customers'][['customer_id', 'customer_state']],
    on='customer_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    tables['order_items'][['order_id', 'product_id', 'price']],
    on='order_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    products_clean[['product_id', 'product_category_name_english']],
    on='product_id',
    how='inner'
)

# group to get monthly revenue, number of unique orders, and average order value
report = merged_df.groupby(['purchase_month', 'product_category_name_english', 'customer_state']).agg(
    revenue=('price', 'sum'),
    number_of_orders=('order_id', 'nunique')
).reset_index()

# add average order value
report['average_order_value'] = (report['revenue'] / report['number_of_orders']).round(2)
report['revenue'] = report['revenue'].round(2)

# rename columns to match what finance wants
report.rename(columns={'product_category_name_english': 'category'}, inplace=True)

display(report.head(15))

,purchase_month,category,customer_state,revenue,number_of_orders,average_order_value
0,2016-09,furniture_decor,RR,72.89,1,72.89
1,2016-09,health_beauty,SP,134.97,1,134.97
2,2016-10,air_conditioning,MG,299.99,1,299.99
3,2016-10,air_conditioning,RS,357.80,1,357.80
4,2016-10,air_conditioning,SC,599.60,1,599.60
5,2016-10,air_conditioning,SP,449.70,2,224.85
6,2016-10,audio,RJ,132.00,1,132.00
7,2016-10,audio,SP,24.99,1,24.99
8,2016-10,auto,AL,49.99,1,49.99
9,2016-10,auto,CE,499.99,1,499.99


### what one row of the final table represents

each row in this table represents the monthly summary of orders and product sales. it tells us for a specific month, product category, and customer state, exactly how many unique orders were placed, the total product revenue generated, and the average order value (excluding shipping costs).

### checking if our aggregation makes sense

to make sure the final table is correct and we didnt lose or double count records during our joins, lets verify that our total revenue from the summary table is close to the raw item prices in the original dataset.

In [63]:
# verify if total prices match original order items (excluding canceled/unavailable orders)
expected_sum = tables['order_items'][tables['order_items']['order_id'].isin(valid_orders['order_id'])]['price'].sum()
actual_sum = report['revenue'].sum()

print(f"Expected revenue sum: {expected_sum:,.2f}")
print(f"Actual revenue sum in report: {actual_sum:,.2f}")
print(f"Difference: {abs(expected_sum - actual_sum):.2f}")

Expected revenue sum: 13,494,400.74
Actual revenue sum in report: 13,494,400.74
Difference: 0.00


### summarizing findings for finance

we found a few things that finance needs to know:
- revenue definition we used the item `price` column from the `order_items` table as the base of product revenue. we did not include `freight_value` because freight is paid for delivery services.

- we filtered out `canceled` and `unavailable` orders. we decided that orders in other intermediate states (like shipped, invoiced, processing) should still count because payment had been initiated or processed, but we can easily restrict this to `delivered` if finance wants a stricter cash basis.

- 98.96% of orders had customer payments matching the order item totals perfectly. about 1% of orders have small discrepancies (like payments exceeding items value or vice-versa), which could be due to voucher gift cards, partial cancellations, or sequence issues. overall, we can trust the figures quite highly.

### what we expect before building

now i am going to join our valid orders with order items, products (using english names), and customer details. this joins them together so we get state and translated category for each order item. then we group everything by month, category, and state to compute total revenue and total order count. we will define revenue as the total product price (since shipping freight value goes to carriers, not direct item sales revenue).

In [64]:
# join tables step by step to build the detailed line items dataset
merged_df = pd.merge(
    valid_orders[['order_id', 'purchase_month', 'customer_id']],
    tables['customers'][['customer_id', 'customer_state']],
    on='customer_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    tables['order_items'][['order_id', 'product_id', 'price']],
    on='order_id',
    how='inner'
)

merged_df = pd.merge(
    merged_df,
    products_clean[['product_id', 'product_category_name_english']],
    on='product_id',
    how='inner'
)

# group to get monthly revenue, number of unique orders, and average order value
report = merged_df.groupby(['purchase_month', 'product_category_name_english', 'customer_state']).agg(
    revenue=('price', 'sum'),
    number_of_orders=('order_id', 'nunique')
).reset_index()

# add average order value
report['average_order_value'] = (report['revenue'] / report['number_of_orders']).round(2)
report['revenue'] = report['revenue'].round(2)

# rename columns to match what finance wants
report.rename(columns={'product_category_name_english': 'category'}, inplace=True)

display(report.head(15))

,purchase_month,category,customer_state,revenue,number_of_orders,average_order_value
0,2016-09,furniture_decor,RR,72.89,1,72.89
1,2016-09,health_beauty,SP,134.97,1,134.97
2,2016-10,air_conditioning,MG,299.99,1,299.99
3,2016-10,air_conditioning,RS,357.80,1,357.80
4,2016-10,air_conditioning,SC,599.60,1,599.60
5,2016-10,air_conditioning,SP,449.70,2,224.85
6,2016-10,audio,RJ,132.00,1,132.00
7,2016-10,audio,SP,24.99,1,24.99
8,2016-10,auto,AL,49.99,1,49.99
9,2016-10,auto,CE,499.99,1,499.99


### what one row of the final table represents

each row in this table represents the monthly summary of orders and product sales. it tells us for a specific month, product category, and customer state, exactly how many unique orders were placed, the total product revenue generated, and the average order value (excluding shipping costs).

### checking if our aggregation makes sense

to make sure the final table is correct and we didnt lose or double count records during our joins, lets verify that our total revenue from the summary table is close to the raw item prices in the original dataset.

In [65]:
# verify if total prices match original order items (excluding canceled/unavailable orders)
expected_sum = tables['order_items'][tables['order_items']['order_id'].isin(valid_orders['order_id'])]['price'].sum()
actual_sum = report['revenue'].sum()

print(f"Expected revenue sum: {expected_sum:,.2f}")
print(f"Actual revenue sum in report: {actual_sum:,.2f}")
print(f"Difference: {abs(expected_sum - actual_sum):.2f}")

Expected revenue sum: 13,494,400.74
Actual revenue sum in report: 13,494,400.74
Difference: 0.00


### summarizing findings for finance

we found a few things that finance needs to know:
1. **revenue definition**: we used the item `price` column from the `order_items` table as the base of product revenue. we did not include `freight_value` because freight is paid for delivery services.
2. **order exclusions**: we filtered out `canceled` and `unavailable` orders. we decided that orders in other intermediate states (like shipped, invoiced, processing) should still count because payment had been initiated or processed, but we can easily restrict this to `delivered` if finance wants a stricter cash basis.
3. **trusting the figures**: 98.96% of orders had customer payments matching the order item totals perfectly. about 1% of orders have small discrepancies (like payments exceeding items value or vice-versa), which could be due to voucher gift cards, partial cancellations, or sequence issues. overall, we can trust the figures quite highly.

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed. Please insert as many cells as you need to complete your code - you do not need to add it all to one cell.*

### checking data consistency on payments and values

lets do one more deep check to see how much payment mismatch exists specifically on completed/delivered orders, to ensure we dont have phantom payments skewing things. we'll group discrepancies to see if they are biased or just random noise.

In [66]:
# match payments only on the filtered valid_orders to isolate the mismatch issue
valid_order_ids = set(valid_orders['order_id'])
valid_comparison = comparison[comparison['order_id'].isin(valid_order_ids)].copy()

# check stats of the difference
print(valid_comparison['diff'].describe())
print(f"\nTotal net discrepancy amount: {valid_comparison['diff'].sum():,.2f}")

count    98198.000000
mean         0.028905
std          1.128854
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: diff, dtype: float64

Total net discrepancy amount: 2,838.38


## 5. Prompts, Checks and AI notes



1. im working in Google Colab on a data cleaning task. The tables are in a dictionary called tables (pandas DataFrames: orders, order_items, payments, reviews, customers, products, sellers, category_translation). A DuckDB connection con and a helper q(sql) already exist. Rules for everything you write: never modify tables[...] directly, always work on a copy named <table>_clean; after creating each clean table register it with con.register("<table>_clean", df); keep code short with a comment on each step; print row counts before and after every step; do only what I ask in each message and nothing more.

2. Please proceed with data integrity check and cleaning for all 8 tables, ensure i perform anylytics engineering approach

3. - check the matching payments against item prices to see the sum of item

4. lets proceed w buildinf to join our valud orders w order item, products, cust details. let me know what the final ytabvle represents

5. check rn if our aggregation make sense? do lmk what doesnt and how we can correct them

6. okay cool now lets summarize tge findin=gs for finance. i want to get the revenue def, order  defand trusting of the figures rates


## 6. What I could not verify

here is a list of things i couldntt fully verify or prove with the datasets we got

1. voucher tracking. a lot of the mismatches between payment value and product cost + freight seems to come from `payment_type = 'voucher'`. because we dont have a loyalty or voucher issues log, i couldn't verify if these are internal discount codes, store credit refunds from previous bad orders, or third-party gift cards. if they are store credits from previous returns, the actual cash revenue might be overstated because we are counting the replacement order but didn't subtract the previous return.

2. customer physical locations. we are using the billing or registry state in `customers` table to represent where the order went. but we don't have shipping address records (like receiver's state) for individual orders. if someone bought a gift for a friend in another state, our geographic revenue report is assigning that money to the buyer's state instead of where the goods actually went.

3. actual cancellation timings. we filtered out orders with status `canceled` or `unavailable`. but some order items might have been canceled *after* shipping or partially refunded without the main order status being changed to `canceled`. since we only have single status stamps and don't have an order log/history table, we can't tell if an order was returned later or refunded manually through customer support.

4. seller payout vs marketplace cut. we don't know the marketplace commission rate (the percentage olist takes). our report calculates total product sales, but we cannot verify how much of this revenue actually stays in the platform's bank account versus how much is paid out to third-party sellers. the `payments` table shows what customers paid, but not what was paid to sellers.